# 2. Exact phase estimation and product formulas
Rebuild Figs. 10 and 11 from two small sector Hamiltonians and the 18 data points of Fig. 13 from the ordered Pauli coefficients of $^{6}$He. Energies are in MeV and evolution times in $\mathrm{MeV}^{-1}$. The two examples use different Hilbert spaces. Phase estimation uses the symmetry sector of the prepared determinant. The product-formula benchmark takes the operator norm in the full six-qubit space.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
if not (DATA / "structural_counts.tsv").is_file():
    raise FileNotFoundError("Start the notebook with its working directory at the repository root.")
sys.path.insert(0, str(ROOT / "lib"))

def read_data(filename):
    return json.loads((DATA / filename).read_text())

## Figs. 10 and 11 · evolution and inverse QFT
For an eigenenergy $E_j$, ancilla time index $t$ acquires the phase $\exp[-it\,\delta t(E_j-E_{\rm shift})]$. The amplitude at register bin $k$ is
$$a_{kj}=\frac{1}{L}\sum_{t=0}^{L-1}e^{-it\delta t(E_j-E_{\rm shift})}e^{-2\pi i kt/L},\qquad L=2^{N_a}.$$
For the prepared determinant $|d\rangle$, the measured probability is $\sum_j |\langle E_j|d\rangle|^2|a_{kj}|^2$. A phase between two integer register bins gives nonzero weight farther away. Inspect the arrays below or edit the register size in the exploration cell.

In [ ]:
def qpe_bins(sector, delta_t, ancillas):
    """Exact sector evolution followed by the inverse QFT, with little-endian output bins."""
    energies, vectors = np.linalg.eigh(np.array(sector["sector_hamiltonian_MeV"]))
    initial = sector["basis"].index(sector["initial_bitmask"])
    weights = np.abs(vectors[initial, :])**2
    shift = sector["energy_shift_MeV"]
    phases = -(energies - shift)*delta_t/(2*np.pi)
    if not np.all((0 <= phases) & (phases < 1)):
        raise ValueError("The shifted spectrum lies outside the chosen energy window.")
    size = 2**ancillas
    bins = np.arange(size)
    phases_in_time = np.exp(2j*np.pi*np.outer(bins, phases))
    amplitudes = np.fft.fft(phases_in_time, axis=0)/size
    probabilities = (np.abs(amplitudes)**2) @ weights
    return energies, shift - 2*np.pi*bins/(size*delta_t), probabilities

In [ ]:
examples = read_data("demonstrations.json")["qpe"]
sectors = read_data("qpe_sectors.json")
for example in examples:
    sector = sectors[example["target"]]
    energies, bins, p = qpe_bins(sector, example["delta_t"], example["n_ancilla"])
    np.testing.assert_allclose(energies, example["Ens_exact"], atol=1e-12)
    np.testing.assert_allclose(np.column_stack((bins, p)), example["Energy_list"], atol=1e-12)
    np.testing.assert_allclose(p.sum(), 1.0, atol=1e-13)
    print(f"{example['nuc']}: {len(energies)} sector eigenvalues, {len(p)} probabilities verified.")

In [ ]:
example = examples[1]
ancillas = 6
_, energy_grid, probability = qpe_bins(sectors[example["target"]], example["delta_t"], ancillas)
pd.DataFrame({"bin": np.arange(len(probability)), "energy_MeV": energy_grid,
              "probability": probability}).sort_values("probability", ascending=False).head(10)

## Fig. 13 · product-formula error on six qubits
Remove the constant identity shift, form the full $64\times64$ Hamiltonian and exponentiate it. For each of the published step counts, apply the ordered first-, second- or fourth-order Suzuki product and compute the spectral norm of its difference from exact evolution. The Pauli coefficients retain the original order. Each Pauli row is stored as a signed permutation to make left multiplication inexpensive.

In [ ]:
def independent_sequence(order, time, coefficients):
    """Chronological rotations, exp(-i * theta * P)."""
    if order == 1:
        return [(j, time * c) for j, c in enumerate(coefficients)]
    if order == 2:
        half = [(j, time * c / 2) for j, c in enumerate(coefficients[:-1])]
        return half + [(len(coefficients) - 1, time * coefficients[-1])] + half[::-1]
    if order == 4:
        a = 1 / (4 - 4 ** (1 / 3))
        return sum((independent_sequence(2, time * s, coefficients)
                    for s in [a, a, 1 - 4 * a, a, a]), [])
    raise ValueError("This check supports orders 1, 2 and 4.")

In [ ]:
from scipy.linalg import expm

benchmark = read_data("trotter_order.json")
settings, case = benchmark["settings"], benchmark["cases"][0]
terms = case["ordered_pauli_coefficients_MeV"]
time = case["time_MeV_inverse"]
coefficients = np.array([c for label, c in terms])
bits = np.arange(2**case["qubits"])
identity = np.eye(len(bits), dtype=complex)
H = np.zeros_like(identity)
permutations, phases = [], []
for label, c in terms:
    x = int("".join("1" if p in "XY" else "0" for p in label), 2)
    z = int("".join("1" if p in "YZ" else "0" for p in label), 2)
    source = bits ^ x
    phase = 1j**label.count("Y") * np.array([(-1)**int(i & z).bit_count() for i in source])
    permutations.append(source)
    phases.append(phase[:, None])
    H[bits, source] += c*phase
np.testing.assert_allclose(H, H.conj().T)
exact = expm(-1j*time*H)
records = []
for order in settings["orders"]:
    for steps in settings["steps"]:
        sequence = independent_sequence(order, time/steps, coefficients)
        step = identity.copy()
        for j, theta in sequence:
            step = np.cos(theta)*step - 1j*np.sin(theta)*phases[j]*step[permutations[j]]
        error = np.linalg.norm(np.linalg.matrix_power(step, steps) - exact, ord=2)
        records.append((order, steps, len(sequence)*steps, error))
measured = pd.DataFrame(records, columns=["order", "steps", "pauli_rotations", "operator_norm_error"])
reference = pd.DataFrame(case["results"])
np.testing.assert_array_equal(measured.iloc[:, :3], reference[measured.columns[:3]])
np.testing.assert_allclose(measured.operator_norm_error, reference.operator_norm_error, rtol=2e-6, atol=2e-12)
for order in settings["orders"]:
    rows = measured[(measured.order == order) & measured.steps.isin(settings["fit_steps"])]
    power = np.polyfit(np.log(rows.steps), np.log(rows.operator_norm_error), 1)[0]
    np.testing.assert_allclose(power, case["fits"][str(order)]["power"], atol=1e-4)
    print(f"Order {order}: fitted power {power:.4f}")
measured